In [ ]:
pip install chromadb

In [ ]:
import chromadb

# client = chromadb.HttpClient(
#     host="api.trychroma.com",
#     port=443,
#     ssl=False,
#     headers={"Authorization": f"Bearer ck-3ywBSArvdWjXJcA4nhxLuzqtBbteSeaZrv6JcobPDjeF",
#              "X-Chroma-Tenant": "6b185675-deb1-46e4-a915-fa4d16da2bd5",
#              "X-Chroma-Database": "sunilTextImage"}
# )

client = chromadb.CloudClient(
  api_key='ck-3ywBSArvdWjXJcA4nhxLuzqtBbteSeaZrv6JcobPDjeF',
  tenant='6b185675-deb1-46e4-a915-fa4d16da2bd5',
  database='sunilTextImage'
)

client

In [ ]:
pip install pymupdf sentence-transformers chromadb pillow

In [ ]:
import os
from typing import List, Dict, Any

import fitz # PyMuPDF
from PIL import image
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb

In [ ]:
PDF_PATH = r"attention is all you need.pdf"   # path to your PDF (same folder as script/notebook)
FIGURES_DIR = "figures"                       # folder for extracted images

os.makedirs(FIGURES_DIR, exist_ok=True)
FIG_COLLECTION_NAME = "attention_figures"
TEXT_COLLECTION_NAME = "attention_text_chunks"
clip_model = SentenceTransformer('clip-ViT-B-32')
#512

In [ ]:
def embed_image(path: str) -> np.ndarray:
    img = Image.open(path).convert("RGB")
    emb = clip_model.encode([img], convert_to_numpy=True, show_progress_bar=False)
    return emb[0]

def embed_text(text: str) -> np.ndarray:
    emb = clip_model.encode([text], convert_to_numpy=True, show_progress_bar=False)
    return emb[0]

In [ ]:
doc = fitz.open(PDF_PATH)
for i in doc:
    print(i.get_text("blocks"))
doc.metadata    

In [ ]:
for i in doc:
    print(i.get_images())

In [ ]:
image_records: List[Dict[str, Any]] = []

print("Extracting images and captions...")
for page_index, page in enumerate(doc, start=1):
    blocks = page.get_text("blocks")  # (x0, y0, x1, y1, text, ...)
    text_blocks = [b for b in blocks if b[4].strip()]  # keep non-empty text blocks

    images = page.get_images(full=True)
    print(f"Page {page_index}: found {len(images)} images")

    for img_index, img in enumerate(images, start=1):
        xref = img[0]

        # Where is this image on the page?
        rects = page.get_image_rects(xref)
        if not rects:
            continue
        rect = rects[0]

        # Heuristic: caption = closest text block BELOW the image
        below_blocks = [b for b in text_blocks if b[1] >= rect.y1]
        candidates = below_blocks if below_blocks else text_blocks

        if candidates:
            closest_block = min(candidates, key=lambda b: abs(b[1] - rect.y1))
            caption_text = closest_block[4].strip().replace("\n", " ")
        else:
            caption_text = ""

        # Extract image binary and save to file
        img_data = doc.extract_image(xref)
        img_bytes = img_data["image"]
        img_ext = img_data.get("ext", "png")

        img_filename = f"page_{page_index:02d}_img_{img_index}.{img_ext}"
        img_path = os.path.join(FIGURES_DIR, img_filename)

        with open(img_path, "wb") as f:
            f.write(img_bytes)

        image_records.append({
            "id": f"p{page_index}_img{img_index}",
            "page_number": page_index,
            "bbox_x0": float(rect.x0),
            "bbox_y0": float(rect.y0),
            "bbox_x1": float(rect.x1),
            "bbox_y1": float(rect.y1),
            "caption": caption_text,
            "image_path": img_path,
        })

print(f"Total extracted images: {len(image_records)}")

In [ ]:
text_records: List[Dict[str, Any]] = []

print("Extracting text chunks...")
MIN_CHARS = 80  # ignore very tiny text fragments

for page_index, page in enumerate(doc, start=1):
    blocks = page.get_text("blocks")  # (x0, y0, x1, y1, text, ...)
    for block_idx, b in enumerate(blocks):
        text = b[4].strip()
        if len(text) < MIN_CHARS:
            continue

        x0, y0, x1, y1 = b[0], b[1], b[2], b[3]

        text_records.append({
            "id": f"p{page_index}_block{block_idx}",
            "page_number": page_index,
            "bbox_x0": float(x0),
            "bbox_y0": float(y0),
            "bbox_x1": float(x1),
            "bbox_y1": float(y1),
            "text": text,
        })

print(f"Total extracted text chunks: {len(text_records)}")

In [ ]:
text_records

In [ ]:
image_records

In [ ]:
fig_collection = client.get_or_create_collection(
    name=FIG_COLLECTION_NAME,
    metadata={"type": "image", "source": "attention_pdf"},
)

text_collection = client.get_or_create_collection(
    name=TEXT_COLLECTION_NAME,
    metadata={"type": "text", "source": "attention_pdf"},
)


In [ ]:
fig_ids = []
fig_embeddings = []
fig_metadatas = []
fig_documents = []

for rec in image_records:
    img_emb = embed_image(rec["image_path"])

    fig_ids.append(rec["id"])
    fig_embeddings.append(img_emb.tolist())
    fig_metadatas.append({
        "page_number": int(rec["page_number"]),
        "bbox_x0": rec["bbox_x0"],
        "bbox_y0": rec["bbox_y0"],
        "bbox_x1": rec["bbox_x1"],
        "bbox_y1": rec["bbox_y1"],
        "image_path": rec["image_path"],
        "caption": rec["caption"],
    })
    fig_documents.append(
        f"Image from 'Attention Is All You Need' on page {rec['page_number']}. "
        f"Caption: {rec['caption']}"
    )

if fig_ids:
    fig_collection.add(
        ids=fig_ids,
        embeddings=fig_embeddings,
        metadatas=fig_metadatas,
        documents=fig_documents,
    )

In [ ]:
txt_ids = []
txt_embeddings = []
txt_metadatas = []
txt_documents = []

for rec in text_records:
    emb = embed_text(rec["text"])

    txt_ids.append(rec["id"])
    txt_embeddings.append(emb.tolist())
    txt_metadatas.append({
        "page_number": int(rec["page_number"]),
        "bbox_x0": rec["bbox_x0"],
        "bbox_y0": rec["bbox_y0"],
        "bbox_x1": rec["bbox_x1"],
        "bbox_y1": rec["bbox_y1"],
    })
    txt_documents.append(rec["text"])

if txt_ids:
    text_collection.add(
        ids=txt_ids,
        embeddings=txt_embeddings,
        metadatas=txt_metadatas,
        documents=txt_documents,
    )

In [ ]:
def answer_query(
    query: str,
    top_k_text: int = 5,
    top_k_img: int = 5,
    top_k_overall: int = 8,
) -> List[Dict[str, Any]]:
    """
    Search both text chunks and images in Chroma Cloud,
    merge results by distance, and return/print top hits.
    """
    q_emb = embed_text(query)

    combined: List[Dict[str, Any]] = []

    # --- Text search ---
    text_results = text_collection.query(
        query_embeddings=[q_emb.tolist()],
        n_results=top_k_text,
        include=["metadatas", "documents", "distances"],
    )

    for meta, doc, dist in zip(
        text_results["metadatas"][0],
        text_results["documents"][0],
        text_results["distances"][0],
    ):
        combined.append({
            "type": "text",
            "distance": float(dist),
            "page_number": meta["page_number"],
            "bbox": (
                meta["bbox_x0"],
                meta["bbox_y0"],
                meta["bbox_x1"],
                meta["bbox_y1"],
            ),
            "content": doc,
        })

    # --- Image search ---
    fig_results = fig_collection.query(
        query_embeddings=[q_emb.tolist()],
        n_results=top_k_img,
        include=["metadatas", "documents", "distances"],
    )

    for meta, doc, dist in zip(
        fig_results["metadatas"][0],
        fig_results["documents"][0],
        fig_results["distances"][0],
    ):
        combined.append({
            "type": "image",
            "distance": float(dist),
            "page_number": meta["page_number"],
            "image_path": meta["image_path"],
            "caption": meta.get("caption", ""),
            "content": doc,
        })

    # Sort by similarity (lower distance = more similar)
    combined.sort(key=lambda x: x["distance"])
    combined = combined[:top_k_overall]

    # Pretty-print
    print(f"\n=== Query: {query} ===")
    for rank, item in enumerate(combined, start=1):
        print(f"\nRank {rank} | type: {item['type']} | distance: {item['distance']:.4f}")
        print("  Page:", item["page_number"])
        if item["type"] == "text":
            preview = item["content"].replace("\n", " ")
            if len(preview) > 260:
                preview = preview[:260] + "..."
            print("  Text:", preview)
        else:
            print("  Image path:", item["image_path"])
            print("  Caption   :", item.get("caption", ""))

    return combined

In [ ]:
answer_query("transfermer model architecture",top_k_text=2, top_k_img=1, top_k_overall=3)